# NCM Omni — direct feed, one arm, no shards

Trains straight off the live Hub stream. There is **no produce step**, no `run/stream`, no `DONE`
marker, no `run/cfg/*.json`, and no arm matrix — one model, one loss, one path from data to
weights.

* `omni.livestream.iter_rows` yields encoded rows in-process; they go into the trainer as they
  arrive, so nothing is ever written to disk to be re-read.
* Held-out validation uses the first 64 rows pulled from the feed, kept in memory (`RowsSet`).
  Stable within a run; it is not an external test set.
* Preflight runs the synthetic copy-relation gate in-process — that is the only way to see
  "the head learns" in 300 steps, because real data cannot drop 1.5 nats that fast. It is still a
  direct feed: no shards, no cache.
* The first line of the training cell is the real-path proof: it prints at step 0 off real data.

Rate warning: the live stream measures ~1.5 samples/s per process, so this is a **correctness and
plumbing** run, not a throughput run. Sharding is the fast path; it is just not the reliable one.


## 1. Credentials — dotenv first, then Colab secrets, then the process env

In [ ]:
from dotenv import load_dotenv
import os

load_dotenv()                                            # a .env next to the notebook
load_dotenv(os.path.expanduser("~/Documents/.env"))      # the local vault, if present
try:                                                     # Colab secrets
    from google.colab import userdata
    for _k in ("HF_TOKEN", "GITHUB_TOKEN"):
        if not os.environ.get(_k):
            try:
                os.environ[_k] = userdata.get(_k)
            except Exception:
                pass
except ImportError:
    pass

print({k: bool(os.environ.get(k)) for k in ("HF_TOKEN", "GITHUB_TOKEN")})  # presence only
assert os.environ.get("HF_TOKEN"), "no HF_TOKEN: set it as a Colab secret or in .env"

## 2. Find the code — open repo, uploaded archive, or a clone

In [ ]:
import glob, os, sys

def _walk_up(start, marker=("omni", "model.py")):
    d = os.path.abspath(start)
    while True:
        if os.path.exists(os.path.join(d, *marker)):
            return d
        up = os.path.dirname(d)
        if up == d:
            return None
        d = up

PKG = _walk_up(os.getcwd())

if PKG is None:                                   # uploaded archive next to the notebook
    for a in (sorted(glob.glob("/content/ncm_omni_v2*.tgz"))
              + sorted(glob.glob("/content/drive/MyDrive/ncm_omni_v2*.tgz"))
              + sorted(glob.glob("/content/ncm_omni_v2*.zip"))):
        import subprocess
        subprocess.run(["tar", "xzf", a, "-C", "/content"] if a.endswith(".tgz")
                       else ["unzip", "-oq", a, "-d", "/content"], check=False)
        PKG = _walk_up("/content")
        if PKG:
            break

if PKG is None:                                    # clone, token via GIT_ASKPASS, never argv
    import stat, subprocess, tempfile
    tok = os.environ.get("GITHUB_TOKEN", "")
    f = tempfile.NamedTemporaryFile("w", suffix=".sh", delete=False)
    f.write('#!/bin/sh\ncase "$1" in *sername*) echo x-access-token ;; *) echo "$NCM_TOK" ;; esac\n')
    f.close()
    os.chmod(f.name, stat.S_IRWXU)
    env = dict(os.environ, GIT_ASKPASS=f.name, NCM_TOK=tok, GIT_TERMINAL_PROMPT="0")
    subprocess.run(["git", "clone", "--depth", "1",
                    "https://github.com/adi274903/ncm-omni-v2.git", "/content/ncm_omni_v2"],
                   env=env, check=True)
    os.unlink(f.name)
    PKG = "/content/ncm_omni_v2"

os.chdir(PKG)
sys.path.insert(0, PKG)
os.environ["PYTHONPATH"] = PKG + os.pathsep + os.environ.get("PYTHONPATH", "")
print("PKG:", PKG)

## 3. Hardware — the step is meaningless on CPU

In [ ]:
import torch
from omni.train import fast_math

print("torch", torch.__version__, "| cuda", torch.cuda.is_available())
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print(f"{p.name} | {p.total_memory / 1e9:.1f} GB | sm {p.major}.{p.minor}")
    print("fast math:", fast_math())
else:
    print("!! no GPU: Runtime -> Change runtime type -> A100")
assert torch.cuda.is_available(), "direct-feed training needs CUDA here"

## 4. Config — built in memory, never written to disk

`build_parser` + `apply_overrides` is the same code the CLI uses, so the notebook cannot drift
from the command line. `train_cfg` flattens it into what the trainer expects (the same function
`write_cfgs` uses for the shard path).

`MODE = "strict"` is the only arm: every code of a level is emitted in **one** forward, 6 forwards
for 6 levels instead of 504. No `block`, no `ar`, no A/B.


In [ ]:
import os
import run_omni as R

STEPS, BATCH, LEN, EVAL_EVERY, SMOKE = 300, 64, 64, 100, 0   # SMOKE=20 for a timing-only run
MODE = "strict"                                             # the single arm

args = R.build_parser().parse_args([
    "train", "--preset", "md", "--base-len", str(LEN), "--batch", str(BATCH)])
rt = R.apply_overrides(R.default_cfg(), args)
cfg = R.train_cfg(rt, MODE, out_dir=os.path.join(PKG, "run", "direct"), steps=STEPS)
cfg["eval_every"] = EVAL_EVERY
cfg["eval_files"] = {}                      # no npz anywhere: validation is the in-memory holdout
cfg["preflight"] = dict(steps=300, batch=8, mask_mode=MODE)

print("model", cfg["model"]["n_embd"], "x", cfg["model"]["n_layer"],
      "| mask", cfg["model"]["mask_mode"], "| batch", cfg["batch"], "| steps", cfg["steps"])
print("out", cfg["out_dir"])

## 5. TensorBoard — started before the first step

In [ ]:
%load_ext tensorboard
%tensorboard --logdir /content/ncm_omni_v2/run/direct/tb

## 6. The direct feed

`iter_rows` pulls from the Hub, decodes with the frozen codecs and yields encoded rows. We take
the first 64 as the holdout and train on everything after them — no files, no re-read.


In [ ]:
import itertools
from omni.livestream import iter_rows

live = dict(rt["produce"])
live.update(sched=rt["sched"], vocab=rt["vocab"], batch_docs=8)

HOLD = 64
raw = iter_rows(live, verbose=True)
hold = [next(raw) for _ in range(HOLD)]
feed = raw                                   # the 64 holdout rows are already consumed

print(f"holdout {len(hold)} rows | mix {live['mix']} | sources {len(live.get('sources') or {}) or 'registry'}")

## 7. Preflight — the gate, in-process

Same synthetic copy-relation stream as always: 300 steps at batch 8 must drop every level `k >= 1`
by at least 1.5 nats and land under `0.5 * ln(V)`. It runs on a direct feed (batches built in the
cell, never on disk) and its LR is scaled from the preset rather than hardcoded.

This is the check that catches a broken head in 50 seconds instead of 300 wasted GPU-hours.


In [ ]:
from omni.preflight import run_preflight

res = run_preflight(cfg)
print("PREFLIGHT", "PASSED" if res["passed"] else "FAILED",
      "| failing:", res.get("failing") or "none")

## 8. Train — one arm, off the feed

`run(cfg, feed=..., eval_rows=...)` bypasses `ShardStream` entirely. It returns normally, so the
kernel survives; there is no `os._exit` on this path. Step 0 logs immediately — that line is the
proof the real data path works.


In [ ]:
cfg["preflight"]["enabled"] = False      # gated in the cell above; do not run it twice
from omni.train import run

run(cfg, smoke=SMOKE or None, feed=feed, eval_rows=hold)

## 9. What came out

In [ ]:
import json, os

path = os.path.join(cfg["out_dir"], "metrics.jsonl")
rows = [json.loads(l) for l in open(path) if l.strip()]
train = [r for r in rows if not r.get("val")]
val = [r for r in rows if r.get("val")]

last = train[-1]
print(f"steps {len(train)} | loss {train[0]['loss']:.4f} -> {last['loss']:.4f} "
      f"| {last['pos_per_s']:,.0f} pos/s | {last['tflops']} TFLOPS")
for r in val[-3:]:
    print(f"  step {r['step']:>5}  val {r['loss']:.4f}")
print("tb:", os.path.join(cfg["out_dir"], "tb"))